## Protein and Ligand Prep

In [1]:
import osprey
osprey.start()
import osprey.prep

OSPREY 3.3-dev, Python 3.10.15, Java 17.0.2, Linux-5.14.0-503.14.1.el9_5.x86_64-x86_64-with-glibc2.31
Using up to 1024 MiB heap memory: 128 MiB for garbage, 896 MiB for storage


In [2]:
osprey.__file__

'/hpc/group/biostat/etm33/miniconda3/envs/AmberTools22/lib/python3.10/site-packages/osprey/__init__.py'

In [3]:
import os
os.getcwd()

'/hpc/home/etm33/kinase_inhibitor_design'

In [4]:
pdb_path = './structures/processed/pkn2_dephos.pdb'
pdb = osprey.prep.loadPDB(open(pdb_path, 'r').read())

In [5]:
print('Loaded %d molecules:' % len(pdb))
for mol in pdb:
    print('\t%s: %s' % (mol, osprey.prep.molTypes(mol)))

Loaded 2 molecules:
	Chain A: [Protein]
	Chain E: [Protein]


In [6]:
# looks like the PDB file has a protein chain and a small molecule
# the protein chain must be PTPase, and the small molecule must be HEPES
# (ignore the solvent molecules, if any)
target = pdb[0]
ligand = pdb[1]
mols = [target, ligand]

In [7]:
# start the local service that calls AmberTools for us
# NOTE: this will only work on Linux machines
with osprey.prep.LocalService():

    # Molecule Preparation Step 1: remove duplicate atoms
    # Duplicate atoms don't usually appear in files from the PDB,
    # but these errors can happen sometimes in modified PDB files.
    for mol in mols:
        # remove all but the first duplicated atom from each group
        for group in osprey.prep.duplicateAtoms(mol):
            for atomi in range(1, len(group.getAtoms())):
                group.remove(atomi)
                print('removed duplicate atom %s' % group)
    
    # Molecule Preparation Step 2: add missing heavy atoms
    # Somtimes atom positions are not well-resolved in the electron density,
    # or protein chain end-caps are missing.
    # But we still want to include these atoms in the molecular models to
    # be able to infer bonds correctly in the next step.
    for mol in mols:
        for missing_atom in osprey.prep.inferMissingAtoms(mol):
            missing_atom.add()
            print('added missing atom: %s' % missing_atom)

    # Molecule Preparation Step 3: add bonds
    # PDB files contain no explicit information about bonds, so we have to
    # infer where they might be based on the atoms we can see.
    for mol in mols:
        bonds = osprey.prep.inferBonds(mol)
        for bond in bonds:
            mol.getBonds().add(bond)
        print('added %d bonds to %s' % (len(bonds), mol))

    for mol in mols:
        protonated_atoms = osprey.prep.inferProtonation(mol)
        for protonated_atom in protonated_atoms:
            protonated_atom.add()
        print('added %d hydrogens to %s' % (len(protonated_atoms), mol))

    # Moleclue Preparation Step 7: save the results
    ligand_path = './structures/processed/pkn2_dephos-ligand.pdb'
    target_path = './structures/processed/pkn2_dephos-target.pdb'
    open(ligand_path, 'w').write(osprey.prep.savePDB(ligand))
    print('saved prepared PDB to %s' % ligand_path)
    open(target_path, 'w').write(osprey.prep.savePDB(target))
    print('saved prepared PDB to %s' % target_path)

print('Molecule preparation complete!')

18:39:45.348 [main] INFO  ktor.application - Autoreload is disabled because the development mode is off.
18:39:45.391 [main] INFO  ktor.application - Responding at http://0.0.0.0:44342
Osprey prep local service started


log4j:WARN No appenders could be found for logger (org.apache.http.impl.nio.client.MainClientExec).
log4j:WARN Please initialize the log4j system properly.


added missing atom: CG @ A651
added missing atom: CD @ A651
added missing atom: OE1 @ A651
added missing atom: NE2 @ A651
added missing atom: CE @ A905
added missing atom: NZ @ A905
added missing atom: CD @ A928
added missing atom: CE @ A928
added missing atom: NZ @ A928
added missing atom: CD @ A930
added missing atom: CE @ A930
added missing atom: NZ @ A930
added missing atom: NE @ A938
added missing atom: CZ @ A938
added missing atom: NH1 @ A938
added missing atom: NH2 @ A938
added missing atom: CZ @ A964
added missing atom: NH1 @ A964
added missing atom: NH2 @ A964
added missing atom: CG @ A972
added missing atom: CD @ A972
added missing atom: OE1 @ A972
added missing atom: OE2 @ A972
added missing atom: CE @ A973
added missing atom: CG @ A975
added missing atom: CD @ A975
added missing atom: NE @ A975
added missing atom: CZ @ A975
added missing atom: NH1 @ A975
added missing atom: NH2 @ A975
added missing atom: OXT @ A984
added missing atom: OXT @ E961
added 2786 bonds to Chain A


## Merge Protonated Protein and Ligand in PDB

Used PyMol

```
load ~/Downloads/pkn2_dephos-target.pdb
load ~/Downloads/pkn2_dephos-ligand.pdb
```

Saved the entire molecule-- note that there are clashes!

```
pkn2_dephos-complex.pdb
```

## KStar Prep

In [8]:
# BioPython throws errors that aren't really errors, so we'll ignore
import warnings
warnings.simplefilter('ignore')

In [9]:
os.getcwd()

'/hpc/home/etm33/kinase_inhibitor_design'

In [10]:
# choose a forcefield
ffparams = osprey.ForcefieldParams()

In [11]:
# read a PDB file for molecular info
mol = osprey.readPdb('./structures/processed/pkn2_dephos-complex.pdb')

read PDB file from file: ./structures/processed/pkn2_dephos-complex.pdb


In [12]:
# make sure all strands share the same template library
templateLib = osprey.TemplateLibrary(ffparams.forcefld)

**Clashes (Protein)**

* 817F (12L)

* 667H (7D)

* 786D (5R)

* T751, I749 (2F)

**Clashes (ligand):**

* 1F

* 5R

* 7D

* 12L




In [13]:
# define the protein strand
protein = osprey.Strand(mol, templateLib=templateLib, residues=['A849', 'A858'])
# protein.flexibility['A749'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()
#protein.flexibility['A750'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()
#protein.flexibility['A751'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()

GLU A 849
MET A 850
LEU A 851
VAL A 852
GLY A 853
GLU A 854
SER A 855
PRO A 856
PHE A 857
PRO A 858


In [15]:
# define the ligand strand
ligand = osprey.Strand(mol, templateLib=templateLib, residues=['E948', 'E961'])
ligand.flexibility['E1'].setLibraryRotamers(osprey.WILD_TYPE, 'ALA', 'TYR').addWildTypeRotamers().setContinuous()
ligand.flexibility['E2'].setLibraryRotamers(osprey.WILD_TYPE, 'ALA', 'TYR').addWildTypeRotamers().setContinuous()
ligand.flexibility['E3'].setLibraryRotamers(osprey.WILD_TYPE, 'ALA', 'TYR').addWildTypeRotamers().setContinuous()
ligand.flexibility['E4'].setLibraryRotamers(osprey.WILD_TYPE, 'ALA', 'TYR').addWildTypeRotamers().setContinuous()

PHE E 948
PRO E 949
LEU E 950
LYS E 951
ARG E 952
HIE E 953
ASP E 954
LYS E 955
VAL E 956
ASP E 957
ASP E 958
LEU E 959
SER E 960
LYS E 961


AttributeError: 'NoneType' object has no attribute 'setLibraryRotamers'